# ExaHyPE 2 Tutorials, Part 2: The Shallow-Water Equations

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../../python/"))
import peano4, exahype2

In [ ]:
unknowns = {"h": 1, "hu": 2, "b": 1}
auxiliary_variables = {}
max_h = 1.1 * 2.0 / (3.0 ** 3)
min_h = max_h / 3

my_solver = exahype2.solvers.aderdg.GlobalAdaptiveTimeStep(
    name                  = "SWESolver",
    order                 = 3,
    min_cell_h            = min_h,
    max_cell_h            = max_h,
    time_step_relaxation  = 0.9,
    unknowns              = unknowns,
    auxiliary_variables   = auxiliary_variables
)

In [ ]:
my_solver.set_implementation(
    flux = exahype2.solvers.PDETerms.User_Defined_Implementation,
    ncp  = exahype2.solvers.PDETerms.User_Defined_Implementation,
    refinement_criterion = exahype2.solvers.PDETerms.User_Defined_Implementation,
    max_eigenvalue=exahype2.solvers.PDETerms.User_Defined_Implementation,
)

In [ ]:
exahype2_project = exahype2.Project(
    namespace     = ["tutorials", "exahype2", "swe"],
    directory     = ".",
    project_name  = "DamBreak",
    executable    = "DamBreak"
)

exahype2_project.add_solver(my_solver)
exahype2_project.set_output_path("solutions")

In [ ]:
exahype2_project.set_global_simulation_parameters(
    dimensions            = 2,
    size                  = [ 2.0,  2.0],
    offset                = [-1.0, -1.0],
    min_end_time          = 0.5,
    max_end_time          = 0.5,
    first_plot_time_stamp = 0.0,
    time_in_between_plots = 0.1,
    periodic_BC           = [False, False],
)

In [ ]:
exahype2_project.set_load_balancer("new ::exahype2::LoadBalancingConfiguration")
exahype2_project.set_build_mode(peano4.output.CompileMode.Release)
peano4_project = exahype2_project.generate_Peano4_project()
peano4_project.output.makefile.add_cpp_file("SWESolver.cpp")
peano4_project.output.makefile.add_CXX_flag("-DExercise=1")
peano4_project.generate()

In this example the implementation is not provided for you, so you will need to look into the implementation file **SWESolver.cpp**, implement the functions and re-compile the project before you can run it. Go do so now!

In [ ]:
peano4_project.build(make=True, make_clean_first=False, throw_away_data_after_build=False)

In [ ]:
import subprocess
subprocess.run("./DamBreak", shell = True, executable="/bin/bash")

In [ ]:
env = os.environ.copy()
env["PYTHONPATH"] = "../../../python" + os.pathsep + env.get("PYTHONPATH", "")

import subprocess
subprocess.call(
    [
        "python3", 
        "-m", "visualisation.convert_to_vtu",
        "solutions/solution-SWESolver.peano-patch-file",
        "-j",
    ],
    env=env
)